# 💰 Personal Finance AI Agent — LangChain Capstone

**Goal:** build an AI assistant that answers money questions (EMI, SIP returns, savings goals, budgets) by letting an **LLM choose the right Python tool** — the LLM never does the arithmetic itself.

```
 user question ─► LLM (gpt-4o-mini) ─ decides ─► tool call: name + JSON arguments
                      ▲                                   │
                      │                                   ▼
              final answer  ◄──── tool result ◄──── Python function (exact maths)
```

| Step | What we build | Why it matters |
|---|---|---|
| 0–1 | Environment + API key | Reproducible setup |
| 2 | Formulas | Know what the tools must return |
| 3–7 | Five tools as plain Python functions | Testable without any LLM |
| 8 | Wrap functions as LangChain tools | The LLM only sees names, descriptions and argument schemas |
| 9–10 | Create the agent + a helper that shows which tools it chose | The "agent" = LLM + tools + a loop |
| 11–12 | Run the 5 required queries + 4 extra ones, auto-check tool choice and numbers | Evidence for the capstone |
| 13 | Guardrails: off-topic, missing input, unsafe expression | Real-world robustness |

> Every code cell below is copied from `finance_tools.py`, `agent.py` and `demo_queries.py`, so the notebook and the Streamlit app always run identical logic.

## Step 0 — Environment setup
**What:** install LangChain 1.x, the OpenAI integration and `python-dotenv`.
**Why:** LangChain 1.x provides `create_agent`, the modern one-call way to build a tool-calling agent.
Skip this cell if you already ran `pip install -r requirements.txt` in your virtual environment.

In [ ]:
# Run once. Restart the kernel afterwards if packages were newly installed.
%pip install -q "langchain>=1.0,<2.0" "langchain-openai>=1.0,<2.0" python-dotenv

## Step 1 — Load your OpenAI API key
**What:** read `OPENAI_API_KEY` from a local `.env` file, or ask for it securely.
**Why:** never hard-code or commit keys. `getpass` hides the key while you type it.

In [ ]:
import os, getpass
from dotenv import load_dotenv

load_dotenv()                                   # reads .env if it exists
if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Paste your OpenAI API key: ")
print("API key available:", bool(os.getenv("OPENAI_API_KEY")))

## Step 2 — The formulas the tools must implement
With annual rate $R\%$ and duration $Y$ years, use the **monthly rate** $r = \dfrac{R}{12 \times 100}$ and **months** $n = 12Y$.

| Tool | Formula | Meaning of symbols |
|---|---|---|
| EMI | $EMI = \dfrac{P \, r \, (1+r)^n}{(1+r)^n - 1}$ | $P$ = loan amount |
| SIP future value | $FV = P \cdot \dfrac{(1+r)^n - 1}{r}$ | $P$ = monthly investment |
| Goal SIP (inverse of SIP) | $P = \dfrac{FV \cdot r}{(1+r)^n - 1}$ | $FV$ = target corpus |
| Budget (50-30-20) | Needs 50 % · Wants 30 % · Savings 20 % | of monthly income |
| Calculator | safe arithmetic evaluation | `+ - * / ** % //` and parentheses |

**Hand check you can do on a calculator:** ₹5,000/month, 12 % p.a., 10 years → $r=0.01$, $n=120$, $(1.01)^{120}\approx 3.3004$ → $FV \approx 5000 \times 230.04 \approx$ ₹11.5 lakh. Our function must reproduce this.

## Step 3 — Imports and helpers (Indian number formatting + validation)
**Why:** results like `1150193.45` are hard to read; Indian grouping gives `₹11,50,193.45`. Validation helpers reject nonsense inputs (negative loans, 500 % rates) with clear messages the agent can relay.

In [ ]:
import ast
import json
import operator
import re

from langchain_core.tools import tool
from pydantic import BaseModel, Field


MAX_RATE = 100.0      # sanity cap on any annual percentage rate
MAX_YEARS = 100.0     # sanity cap on any duration


def format_indian_number(value, decimals=2):
    """Indian digit grouping: 1234567.5 -> '12,34,567.50'."""
    value = round(float(value), decimals)
    sign = "-" if value < 0 else ""
    whole, _, frac = f"{abs(value):.{decimals}f}".partition(".")
    if len(whole) > 3:
        head, tail = whole[:-3], whole[-3:]
        groups = []
        while len(head) > 2:
            groups.insert(0, head[-2:])
            head = head[:-2]
        if head:
            groups.insert(0, head)
        whole = ",".join(groups + [tail])
    return f"{sign}{whole}" + (f".{frac}" if decimals else "")


def format_inr(value, decimals=2):
    """Rupee string with Indian grouping: 1150193.8 -> '₹11,50,193.80'."""
    text = format_indian_number(abs(value), decimals)
    return ("-" if value < 0 else "") + "₹" + text


def _check_positive(name, value):
    if isinstance(value, bool) or not isinstance(value, (int, float)) or value <= 0:
        raise ValueError(f"{name} must be a number greater than 0 (got {value!r}).")


def _check_rate(name, value):
    if isinstance(value, bool) or not isinstance(value, (int, float)) or not 0 <= value <= MAX_RATE:
        raise ValueError(f"{name} must be between 0 and {MAX_RATE:g} percent per year (got {value!r}).")


def _to_months(name, years):
    _check_positive(name, years)
    if years > MAX_YEARS:
        raise ValueError(f"{name} must be at most {MAX_YEARS:g} years (got {years!r}).")
    months = round(years * 12)
    if months < 1:
        raise ValueError(f"{name} must be at least 1 month.")
    return months

In [ ]:
print(format_inr(1234567.8), format_inr(100000), format_inr(999))   # expect ₹12,34,567.80  ₹1,00,000.00  ₹999.00

## Step 4 — Tool 1: Calculator (safe)
**What:** evaluate arithmetic written as text, e.g. `"0.20 * 85000"`.
**Why safe:** Python's `eval()` would let an LLM (or a user) run arbitrary code. Instead we parse the text into an AST and allow **only** numbers and arithmetic operators. Huge powers such as `9**9**9` are rejected.

In [ ]:
_BINARY_OPS = {
    ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
    ast.Div: operator.truediv, ast.FloorDiv: operator.floordiv,
    ast.Mod: operator.mod, ast.Pow: operator.pow,
}
_UNARY_OPS = {ast.UAdd: operator.pos, ast.USub: operator.neg}
MAX_EXPRESSION_LENGTH = 200


def _clean_expression(expression):
    """Normalise things users and LLMs commonly type (₹, commas, ^, '20% of')."""
    text = expression.strip().lower()
    text = text.replace("₹", "").replace(",", "")
    text = text.replace("×", "*").replace("÷", "/").replace("^", "**")
    # "20% of 85000" -> "(20/100)*85000"
    text = re.sub(r"(\d+(?:\.\d+)?)\s*%\s*of\s*", r"(\1/100)*", text)
    return text


def _eval_node(node):
    """Evaluate a parsed expression using an explicit whitelist - never eval()."""
    if isinstance(node, ast.Expression):
        return _eval_node(node.body)
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)) \
            and not isinstance(node.value, bool):
        return float(node.value)
    if isinstance(node, ast.BinOp) and type(node.op) in _BINARY_OPS:
        return _BINARY_OPS[type(node.op)](_eval_node(node.left), _eval_node(node.right))
    if isinstance(node, ast.UnaryOp) and type(node.op) in _UNARY_OPS:
        return _UNARY_OPS[type(node.op)](_eval_node(node.operand))
    raise ValueError("Unsupported expression. Use only numbers and + - * / ** % // ( ).")


def safe_calculate(expression):
    """Safely evaluate an arithmetic expression and return a result dictionary."""
    if not isinstance(expression, str) or not expression.strip():
        raise ValueError("expression must be a non-empty string.")
    if len(expression) > MAX_EXPRESSION_LENGTH:
        raise ValueError(f"expression is too long (max {MAX_EXPRESSION_LENGTH} characters).")
    cleaned = _clean_expression(expression)
    try:
        result = _eval_node(ast.parse(cleaned, mode="eval"))
    except SyntaxError:
        raise ValueError(f"Could not parse expression: {expression!r}") from None
    except ZeroDivisionError:
        raise ValueError("Division by zero.") from None
    except OverflowError:
        raise ValueError("Result is too large to compute.") from None
    result = round(result, 10)
    is_whole = abs(result - round(result)) < 1e-9
    shown = format_indian_number(result, 0) if is_whole else format_indian_number(result, 4).rstrip("0").rstrip(".")
    return {"expression": expression, "evaluated_as": cleaned,
            "result": int(round(result)) if is_whole else result, "result_formatted": shown}

In [ ]:
for text in ["0.20 * 85000", "20% of ₹85,000", "(1 + 0.08) ** 5", "2 ^ 10"]:
    print(f"{text:20s} -> {safe_calculate(text)['result_formatted']}")

print("\n-- unsafe / invalid inputs must be refused --")
for bad in ["__import__('os').system('dir')", "9**9**9", "1/0", "hello"]:
    try:
        safe_calculate(bad)
    except ValueError as error:
        print(f"{bad:35s} -> blocked: {error}")

## Step 5 — Tool 2: EMI calculator
**What:** monthly instalment, total interest and total repayment for a loan.
**Check:** the classic benchmark — ₹10 lakh at 9 % for 20 years — should give about **₹8,997**. We also prove correctness independently by simulating the loan month by month: after `n` payments the balance must be ≈ ₹0.

In [ ]:
def calculate_emi(loan_amount, annual_interest_rate, tenure_years):
    """EMI = P * r * (1+r)^n / ((1+r)^n - 1)   with r = annual_rate / 12 / 100,  n = years * 12."""
    _check_positive("loan_amount", loan_amount)
    _check_rate("annual_interest_rate", annual_interest_rate)
    n = _to_months("tenure_years", tenure_years)
    r = annual_interest_rate / 12 / 100
    if r == 0:
        emi = loan_amount / n                                  # interest-free loan
    else:
        growth = (1 + r) ** n
        emi = loan_amount * r * growth / (growth - 1)
    total_payment = emi * n
    total_interest = total_payment - loan_amount
    return {
        "loan_amount": round(loan_amount, 2),
        "annual_interest_rate_percent": annual_interest_rate,
        "tenure_months": n,
        "monthly_emi": round(emi, 2),
        "total_interest": round(total_interest, 2),
        "total_payment": round(total_payment, 2),
        "summary": (f"EMI of {format_inr(emi)} per month for {n} months on a loan of "
                    f"{format_inr(loan_amount)} at {annual_interest_rate:g}% p.a. "
                    f"Total interest {format_inr(total_interest)}; total repayment {format_inr(total_payment)}."),
    }

In [ ]:
loan = calculate_emi(1_000_000, 9, 20)
print(loan["summary"])

# Independent verification: simulate the loan month by month
balance, r = 1_000_000.0, 0.09 / 12
for _ in range(loan["tenure_months"]):
    balance = balance * (1 + r) - loan["monthly_emi"]
print("Balance after the last EMI (should be ~0):", round(balance, 2))

## Step 6 — Tool 3: SIP future-value calculator
**What:** future value of a fixed monthly investment, using the formula given in the assignment (investments made at the **end** of each month).
**Check:** ₹5,000 × 12 % × 10 years ≈ ₹11.5 lakh, verified against a month-by-month simulation.

In [ ]:
def calculate_sip_future_value(monthly_investment, annual_return_rate, years):
    """FV = P * ((1+r)^n - 1) / r   with r = annual_rate / 12 / 100,  n = years * 12."""
    _check_positive("monthly_investment", monthly_investment)
    _check_rate("annual_return_rate", annual_return_rate)
    n = _to_months("years", years)
    r = annual_return_rate / 12 / 100
    future_value = monthly_investment * n if r == 0 else monthly_investment * ((1 + r) ** n - 1) / r
    invested = monthly_investment * n
    gains = future_value - invested
    return {
        "monthly_investment": round(monthly_investment, 2),
        "annual_return_rate_percent": annual_return_rate,
        "tenure_months": n,
        "total_invested": round(invested, 2),
        "estimated_gains": round(gains, 2),
        "future_value": round(future_value, 2),
        "summary": (f"Investing {format_inr(monthly_investment)} per month for {n} months at an assumed "
                    f"{annual_return_rate:g}% p.a. grows to about {format_inr(future_value)}: "
                    f"{format_inr(invested)} invested + {format_inr(gains)} estimated gains."),
    }

In [ ]:
sip = calculate_sip_future_value(5000, 12, 10)
print(sip["summary"])

# Independent verification: grow the pot month by month
pot, r = 0.0, 0.12 / 12
for _ in range(sip["tenure_months"]):
    pot = pot * (1 + r) + 5000        # earn this month's interest, then add the instalment
print("Simulated corpus:", round(pot, 2), "| formula:", sip["future_value"])

## Step 7 — Tools 4 & 5: Goal-based SIP and Budget planner
* **Goal SIP** answers *"how much should I save monthly to reach ₹20 lakh in 10 years?"* by inverting the SIP formula. That question does not state a return rate, so the tool defaults to **12 %** and the agent must tell the user it is an assumption.
* **Budget planner** applies the **50-30-20 rule** and adds an *illustrative* split inside each bucket (rent, groceries, SIP, …). The last line of each bucket absorbs rounding so totals always match exactly.

In [ ]:
def calculate_required_sip(target_amount, years, annual_return_rate=12.0):
    """Inverse of the SIP formula:  P = FV * r / ((1+r)^n - 1)."""
    _check_positive("target_amount", target_amount)
    _check_rate("annual_return_rate", annual_return_rate)
    n = _to_months("years", years)
    r = annual_return_rate / 12 / 100
    monthly = target_amount / n if r == 0 else target_amount * r / ((1 + r) ** n - 1)
    invested = monthly * n
    return {
        "target_amount": round(target_amount, 2),
        "annual_return_rate_percent": annual_return_rate,
        "tenure_months": n,
        "required_monthly_investment": round(monthly, 2),
        "total_invested": round(invested, 2),
        "estimated_gains": round(target_amount - invested, 2),
        "summary": (f"To reach {format_inr(target_amount)} in {n} months at an assumed {annual_return_rate:g}% p.a., "
                    f"invest about {format_inr(monthly)} per month "
                    f"({format_inr(invested)} contributed + {format_inr(target_amount - invested)} estimated gains)."),
    }

In [ ]:
goal = calculate_required_sip(2_000_000, 10)           # default 12 % return
print(goal["summary"])
# Round-trip check: investing that amount must give back the target
print("Round trip FV:", calculate_sip_future_value(goal["required_monthly_investment"], 12, 10)["future_value"])

In [ ]:
BUDGET_RULE = {"Needs": 50, "Wants": 30, "Savings": 20}          # the 50-30-20 rule (% of income)
BUDGET_BREAKDOWN = {                                            # illustrative split INSIDE each bucket
    "Needs": {"Rent / home-loan EMI": 50, "Groceries": 20, "Utilities & bills": 10,
              "Transport": 10, "Insurance & health": 10},
    "Wants": {"Dining & entertainment": 40, "Shopping & lifestyle": 35, "Travel & hobbies": 25},
    "Savings": {"SIP / mutual funds": 50, "Emergency fund": 30, "Retirement (PPF / NPS)": 20},
}


def plan_budget(monthly_income):
    """50-30-20 budget: 50% Needs, 30% Wants, 20% Savings, plus an illustrative sub-split."""
    _check_positive("monthly_income", monthly_income)
    buckets = {}
    for bucket, percent in BUDGET_RULE.items():
        amount = round(monthly_income * percent / 100, 2)
        items, running = {}, 0.0
        names = list(BUDGET_BREAKDOWN[bucket])
        for name in names[:-1]:
            value = round(amount * BUDGET_BREAKDOWN[bucket][name] / 100, 2)
            items[name] = value
            running += value
        items[names[-1]] = round(amount - running, 2)           # last line absorbs rounding
        buckets[bucket] = {"percent": percent, "amount": amount, "suggested_breakdown": items}
    return {
        "monthly_income": round(monthly_income, 2),
        "rule": "50-30-20",
        "buckets": buckets,
        "summary": (f"For a monthly income of {format_inr(monthly_income)}: "
                    + ", ".join(f"{b} {format_inr(v['amount'])} ({v['percent']}%)" for b, v in buckets.items()) + "."),
    }

In [ ]:
import json
plan = plan_budget(80_000)
print(plan["summary"], "\n")
for bucket, info in plan["buckets"].items():
    print(f"{bucket} ({info['percent']}%): {format_inr(info['amount'])}")
    for item, amount in info["suggested_breakdown"].items():
        print(f"    - {item:25s} {format_inr(amount)}")

## Step 8 — Wrap the functions as LangChain tools
**What:** the `@tool` decorator turns a function into something an LLM can call.
**This is the heart of tool selection.** The LLM never sees our Python code. It only sees, for each tool:
1. the **name**, 2. the **description** (the docstring) and 3. the **argument schema** (types + field descriptions).

So a good description — *what it does, when to use it, when NOT to use it* — is what makes the LLM pick the correct tool. Notice the calculator's description explicitly says not to use it for EMI/SIP/budget.
Tool functions return JSON text; invalid input returns `{"error": ...}` instead of crashing, so the agent can recover.

In [ ]:
def _run(function, **kwargs):
    """Run a pure function and return JSON text; turn bad input into a readable error the agent can act on."""
    try:
        return json.dumps(function(**kwargs), ensure_ascii=False)
    except ValueError as error:
        return json.dumps({"error": str(error)}, ensure_ascii=False)


class CalculatorInput(BaseModel):
    expression: str = Field(description=(
        "Arithmetic expression using numbers and + - * / ** % // ( ). "
        "Convert percentages to decimals or use 'X% of Y'. Example: '0.20 * 85000'."))


class EMIInput(BaseModel):
    loan_amount: float = Field(description="Principal loan amount in rupees (convert lakh/crore first: 10 lakh = 1000000).")
    annual_interest_rate: float = Field(description="Annual interest rate in percent, e.g. 8.5 for 8.5% p.a.")
    tenure_years: float = Field(description="Loan tenure in years (convert months to years, e.g. 18 months = 1.5).")


class SIPInput(BaseModel):
    monthly_investment: float = Field(description="Amount invested every month, in rupees.")
    annual_return_rate: float = Field(description="Expected annual return in percent, e.g. 12 for 12% p.a.")
    years: float = Field(description="Investment duration in years.")


class GoalSIPInput(BaseModel):
    target_amount: float = Field(description="Target corpus in rupees (convert lakh/crore first: 20 lakh = 2000000).")
    years: float = Field(description="Number of years available to reach the target.")
    annual_return_rate: float = Field(default=12.0, description=(
        "Expected annual return in percent. If the user did not state one, use 12 and tell the user it is an assumption."))


class BudgetInput(BaseModel):
    monthly_income: float = Field(description="Monthly take-home income in rupees.")


@tool("calculator", args_schema=CalculatorInput)
def calculator_tool(expression: str) -> str:
    """General-purpose calculator for plain arithmetic: percentages, sums, differences, products,
    ratios, powers. Use it ONLY for generic maths. Do NOT use it for EMI, SIP / investment growth,
    savings goals or budgets - those have dedicated tools."""
    return _run(safe_calculate, expression=expression)


@tool("emi_calculator", args_schema=EMIInput)
def emi_tool(loan_amount: float, annual_interest_rate: float, tenure_years: float) -> str:
    """Calculate the monthly EMI (equated monthly instalment) of a loan, plus total interest and total
    repayment. Use for home / car / personal loan questions given loan amount, interest rate and tenure."""
    return _run(calculate_emi, loan_amount=loan_amount,
                annual_interest_rate=annual_interest_rate, tenure_years=tenure_years)


@tool("sip_calculator", args_schema=SIPInput)
def sip_tool(monthly_investment: float, annual_return_rate: float, years: float) -> str:
    """Calculate the FUTURE VALUE of a monthly investment (SIP) given the monthly amount, expected annual
    return and duration. Use for 'if I invest X per month for N years at R%, how much will I have?'."""
    return _run(calculate_sip_future_value, monthly_investment=monthly_investment,
                annual_return_rate=annual_return_rate, years=years)


@tool("goal_sip_calculator", args_schema=GoalSIPInput)
def goal_sip_tool(target_amount: float, years: float, annual_return_rate: float = 12.0) -> str:
    """Calculate the MONTHLY amount that must be invested to reach a target corpus in a given number of
    years. Use for 'how much should I save monthly to reach X in N years?'."""
    return _run(calculate_required_sip, target_amount=target_amount, years=years,
                annual_return_rate=annual_return_rate)


@tool("budget_planner", args_schema=BudgetInput)
def budget_tool(monthly_income: float) -> str:
    """Create a monthly budget from income using the 50-30-20 rule (50% needs, 30% wants, 20% savings)
    with a suggested split inside each bucket. Use for 'create a budget / plan my spending' requests."""
    return _run(plan_budget, monthly_income=monthly_income)


ALL_TOOLS = [calculator_tool, emi_tool, sip_tool, goal_sip_tool, budget_tool]

In [ ]:
for t in ALL_TOOLS:
    print(f"TOOL: {t.name}\n  description: {t.description}\n  arguments  : {list(t.args)}\n")

# Calling a tool directly (no LLM) - exactly what the agent does on our behalf:
print(emi_tool.invoke({"loan_amount": 1_000_000, "annual_interest_rate": 8, "tenure_years": 20}))

## Step 9 — Create the agent
**What:** an *agent* = **LLM + tools + system prompt + a loop**. `create_agent` builds that loop:

1. The LLM reads the conversation and either answers or **requests a tool call**.
2. LangChain runs the requested tool and feeds the result back as a `ToolMessage`.
3. The LLM reads the result and either calls another tool or writes the final answer.

**System prompt rules we set:** never calculate yourself; tool-selection guide; convert lakh/crore; ask when inputs are missing (but assume 12 % for the goal tool and say so); keep a fixed answer format; add a returns-aren't-guaranteed note.
**`temperature=0`** keeps tool choice as repeatable as possible.

In [ ]:
import os
from dataclasses import dataclass, field
from typing import Any

from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain_core.messages import AIMessage, HumanMessage, ToolMessage
from langchain_openai import ChatOpenAI

load_dotenv()   # reads OPENAI_API_KEY from a local .env file if present

In [ ]:
DEFAULT_MODEL = "gpt-4o-mini"

SYSTEM_PROMPT = """You are FinBot, a Personal Finance Assistant for Indian users. All money is in Indian Rupees (₹).

GOLDEN RULE: never do arithmetic yourself. Every number you report must come from a tool result.

Choose the tool that matches the question:
- emi_calculator       -> loan EMI (needs loan amount, annual interest %, tenure in years)
- sip_calculator       -> future value of a monthly investment (needs monthly amount, annual return %, years)
- goal_sip_calculator  -> monthly amount needed to reach a target corpus (needs target, years, optional return %)
- budget_planner       -> 50-30-20 monthly budget (needs monthly income)
- calculator           -> any other plain arithmetic, such as percentages of an amount

Rules:
1. Convert Indian units before calling a tool: 1 lakh = 100000, 1 crore = 10000000.
2. If a required input is missing, ask ONE short clarifying question instead of guessing.
   Exception: for goal_sip_calculator, if no return rate is given assume 12% p.a. and say clearly that it is an assumption.
3. Answer format: the headline result first, then 2-4 key figures as bullets, then one line stating the formula or assumption used.
4. For investment projections add one short reminder that returns are assumptions, not guarantees.
5. If a tool returns an error, explain it simply and ask for corrected input.
6. If the question is not about personal finance, politely say you can only help with EMI, SIP, savings goals, budgets and calculations.
This is educational guidance, not professional financial advice."""

In [ ]:
def build_llm(api_key=None, model=DEFAULT_MODEL, temperature=0.0):
    """Create the chat model. temperature=0 keeps tool selection as deterministic as possible."""
    key = api_key or os.getenv("OPENAI_API_KEY")
    if not key:
        raise ValueError("OpenAI API key not found. Set OPENAI_API_KEY (or put it in a .env file).")
    return ChatOpenAI(model=model, temperature=temperature, api_key=key)


def build_agent(api_key=None, llm=None, model=DEFAULT_MODEL):
    """Wire LLM + tools + system prompt into a tool-calling agent (LangChain v1 create_agent)."""
    llm = llm or build_llm(api_key=api_key, model=model)
    return create_agent(model=llm, tools=ALL_TOOLS, system_prompt=SYSTEM_PROMPT)

In [ ]:
agent = build_agent()
print('Agent ready with tools:', [t.name for t in ALL_TOOLS])

## Step 10 — A helper that shows *which tool the LLM chose*
`ask()` runs one question and returns the final **answer** plus a **trace** of every tool call (name, arguments, raw output). The trace is the proof that the LLM — not our code — selected the tool, and we reuse it in the Streamlit app.

In [ ]:
@dataclass
class ToolTrace:
    """One tool call chosen by the LLM."""
    name: str
    args: dict
    output: str = ""


@dataclass
class AgentResult:
    answer: str
    tool_calls: list = field(default_factory=list)


def _as_text(content: Any) -> str:
    """Message content can be a string or a list of content blocks."""
    if isinstance(content, str):
        return content
    parts = []
    for block in content or []:
        parts.append(block.get("text", "") if isinstance(block, dict) else str(block))
    return "".join(parts)


def ask(agent, question, history=None, recursion_limit=12):
    """Send one question (optionally with earlier chat turns) and return the answer + tool trace."""
    messages = list(history or []) + [HumanMessage(content=question)]
    state = agent.invoke({"messages": messages}, config={"recursion_limit": recursion_limit})
    new_messages = state["messages"][len(messages):]

    traces, by_id = [], {}
    for message in new_messages:
        if isinstance(message, AIMessage):
            for call in message.tool_calls:
                trace = ToolTrace(name=call["name"], args=call["args"])
                by_id[call["id"]] = trace
                traces.append(trace)
        elif isinstance(message, ToolMessage) and message.tool_call_id in by_id:
            by_id[message.tool_call_id].output = _as_text(message.content)

    answer = ""
    for message in reversed(new_messages):
        if isinstance(message, AIMessage) and _as_text(message.content).strip():
            answer = _as_text(message.content)
            break
    return AgentResult(answer=answer, tool_calls=traces)

In [ ]:
def show(query, result):
    print("Q:", query)
    for call in result.tool_calls:
        print(f"   🔧 {call.name}({call.args})")
    print("A:", result.answer, "\n" + "-" * 80)

q = "Calculate EMI for ₹10 lakh loan at 8% interest for 20 years"
show(q, ask(agent, q))

## Step 11 — The five required demo queries
We run each query and **automatically check** that the LLM picked the expected tool. Watch how the same SIP tool is chosen for two differently-worded questions, and how a percentage question goes to the calculator instead.

In [ ]:
# (query, tool the agent is expected to choose)
REQUIRED_QUERIES = [
    ("Calculate EMI for ₹10 lakh loan at 8% interest for 20 years", "emi_calculator"),
    ("If I invest ₹7000 monthly for 15 years at 10% return", "sip_calculator"),
    ("Create a budget plan for income ₹90,000", "budget_planner"),
    ("What is 20% of ₹85,000", "calculator"),
    ("How much will I get if I invest ₹3000 per month for 5 years at 8% return", "sip_calculator"),
]

EXTRA_QUERIES = [
    ("If I invest ₹5000 per month for 10 years at 12% return, how much money will I have?", "sip_calculator"),
    ("Calculate EMI for a ₹10 lakh loan at 9% interest for 20 years.", "emi_calculator"),
    ("How much should I save monthly to reach ₹20 lakh in 10 years?", "goal_sip_calculator"),
    ("Create a monthly budget for someone earning ₹80,000.", "budget_planner"),
]

In [ ]:
results = {}          # keep every result for the verification step
rows = []
for number, (query, expected_tool) in enumerate(REQUIRED_QUERIES, start=1):
    results[query] = ask(agent, query)
    show(f"[{number}] {query}", results[query])
    used = [c.name for c in results[query].tool_calls]
    rows.append((number, expected_tool, used, "PASS" if expected_tool in used else "FAIL"))

print(f"{'#':<3}{'expected tool':<22}{'tools used':<35}result")
for number, expected, used, verdict in rows:
    print(f"{number:<3}{expected:<22}{str(used):<35}{verdict}")
print(f"\nTool-selection accuracy: {sum(r[3] == 'PASS' for r in rows)}/{len(rows)}")

## Step 12 — Extra queries from the project brief
These include the **goal-based** question, which gives no return rate — check that the answer states the 12 % assumption.

In [ ]:
rows = []
for number, (query, expected_tool) in enumerate(EXTRA_QUERIES, start=1):
    results[query] = ask(agent, query)
    show(f"[E{number}] {query}", results[query])
    used = [c.name for c in results[query].tool_calls]
    rows.append((number, expected_tool, used, "PASS" if expected_tool in used else "FAIL"))
print(f"Tool-selection accuracy (extra): {sum(r[3] == 'PASS' for r in rows)}/{len(rows)}")

## Step 13 — Verify the numbers *and* the arguments the LLM extracted
Choosing the right tool is half the job; the LLM must also pull the right numbers out of the sentence (10 lakh → 1,000,000; 15 years → 15). We compare the tool output the agent produced against a direct Python call with the correct inputs.

In [ ]:
import json

CHECKS = [   # (query, key in tool output, expected value computed directly in Python)
    (REQUIRED_QUERIES[0][0], "monthly_emi",                 calculate_emi(1_000_000, 8, 20)["monthly_emi"]),
    (REQUIRED_QUERIES[1][0], "future_value",                calculate_sip_future_value(7000, 10, 15)["future_value"]),
    (REQUIRED_QUERIES[3][0], "result",                      17000),
    (REQUIRED_QUERIES[4][0], "future_value",                calculate_sip_future_value(3000, 8, 5)["future_value"]),
    (EXTRA_QUERIES[0][0],    "future_value",                calculate_sip_future_value(5000, 12, 10)["future_value"]),
    (EXTRA_QUERIES[1][0],    "monthly_emi",                 calculate_emi(1_000_000, 9, 20)["monthly_emi"]),
    (EXTRA_QUERIES[2][0],    "required_monthly_investment", calculate_required_sip(2_000_000, 10, 12)["required_monthly_investment"]),
]
good = 0
for query, key, expected in CHECKS:
    output = json.loads(results[query].tool_calls[0].output)
    ok = abs(output.get(key, -1) - expected) < 0.01
    good += ok
    print(f"{'PASS' if ok else 'FAIL'}  {key:28s} agent={output.get(key)}  expected={expected}   <- {query[:55]}")

budget_out = json.loads(results[REQUIRED_QUERIES[2][0]].tool_calls[0].output)
print("PASS " if budget_out["buckets"]["Savings"]["amount"] == 18000 else "FAIL ", "budget for ₹90,000 -> savings ₹18,000")
print(f"\nNumeric verification: {good}/{len(CHECKS)} tool outputs match the reference values")

## Step 14 — Guardrails and edge cases
A production assistant must behave sensibly when users do unexpected things:
1. **Off-topic question** → politely decline, call no tool.
2. **Missing information** → ask a clarifying question instead of inventing numbers.
3. **Unsafe expression** → the calculator refuses to run code.
4. **Invalid input** → the tool returns an error and the agent explains it.

In [ ]:
tests = [
    "Who won the last cricket world cup?",
    "Calculate the EMI for a loan at 9% interest",                 # loan amount and tenure missing
    "Use the calculator to compute __import__('os').listdir('.')", # attempted code execution
    "Calculate EMI for a loan of -5 lakh at 9% for 10 years",      # invalid amount
]
for question in tests:
    show(question, ask(agent, question))

## Step 15 — Conclusion and next steps
**What we proved**
* The LLM selects among five tools from natural-language questions; all arithmetic is deterministic Python verified against independent simulations.
* The agent trace makes every decision auditable (tool, arguments, raw output).
* Guardrails: safe expression parser, input validation, off-topic refusal, clarifying questions.

**Run the app:** `streamlit run app.py`   ·   **Run the tests:** `pytest -q`   ·   **CLI demo:** `python run_demo.py --all`

**Ideas to extend:** step-up SIP, inflation-adjusted goals, tax (80C / LTCG) tool, amortisation-schedule table, conversation memory with a checkpointer, LangSmith tracing, a retrieval tool over RBI / SEBI guidance.

> Educational project — outputs are estimates, not financial advice.